# Day 40: Performance and Reliability Review

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Avoid duplicate preprocessing and ensure the reported version identifies the cached model.

## Theory

For the fitted classifier, the predicted class is the class with maximum predicted probability. Computing probabilities once avoids transforming the same input twice. A version identifier must refer to the loaded object rather than newly changed disk bytes.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Verify full-split parity and model-cache behavior.

In [2]:
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_serving_cache.py","tests/test_contracts.py","tests/test_api.py"],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0,r.stderr

..............                                                           [100%]
============================== warnings summary ===============================
..\venv\Lib\site-packages\fastapi\testclient.py:1
  C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
14 passed, 1 warning in 2.47s



Benchmark actual validated inference for different batch sizes.

In [3]:
import time
import numpy as np
import pandas as pd
from src.dataset import original_split
from src.prediction import predict
X=original_split()[1]
predict(X.head(1))
records=[]
for size in [1,10,130,500]:
    batch=pd.concat([X]*4,ignore_index=True).head(size)
    times=[]
    for _ in range(15):
        start=time.perf_counter()
        out=predict(batch)
        times.append((time.perf_counter()-start)*1000)
        assert len(out)==size
    records.append({"batch_rows":size,"repeats":15,"median_ms":float(np.median(times)),"p95_ms":float(np.percentile(times,95))})
benchmark=pd.DataFrame(records)
benchmark.to_csv("reports/serving_benchmark.csv",index=False)
print(benchmark.to_string(index=False))

 batch_rows  repeats  median_ms   p95_ms
          1       15    10.0994 12.87176
         10       15     9.0592 10.82946
        130       15    10.7219 14.56901
        500       15    15.7078 16.76687


## Findings and interpretation

The complete historical split retains the same class predictions. Model loading and its identity are cached together; request inference now preprocesses once. Measured latency is saved for this environment.

## Limitations

These timings are local measurements, not service-level guarantees or a controlled before/after experiment. Restart the service to load an approved changed artifact.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Avoid duplicate preprocessing and ensure the reported version identifies the cached model.

## Next-day plan

Consolidate the project documentation and architecture.